In [1]:
# ============================================================
# ETAPA 2 - LIMPIEZA Y TRANSFORMACIÓN DE DATOS
# Desafío Profesional Airbnb - Digital House
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ Librerías cargadas")


✅ Librerías cargadas


In [2]:
listings = pd.read_csv("listings.csv", low_memory=False)
calendar = pd.read_csv("calendar.csv")
reviews = pd.read_csv("reviews.csv")

listings["price"] = listings["price"].replace('[\$,]', '', regex=True).astype(float)

print(f"listings:  {listings.shape[0]} filas x {listings.shape[1]} columnas")
print(f"calendar:  {calendar.shape[0]} filas x {calendar.shape[1]} columnas")
print(f"reviews:   {reviews.shape[0]} filas x {reviews.shape[1]} columnas")

listings:  23729 filas x 106 columnas
calendar:  8661286 filas x 7 columnas
reviews:   387099 filas x 6 columnas


In [3]:
cols_100_nulos = ["licence", "thumbnail_url", "xl_picture_url", 
                  "medium_url", "neighbourhood_group_cleansed"]

cols_irrelevantes = [
    "listing_url", "scrape_id", "last_scraped", "picture_url",
    "host_url", "host_thumbnail_url", "host_picture_url",
    "calendar_last_scraped", "requires_license"
]

cols_a_eliminar = [c for c in cols_100_nulos + cols_irrelevantes if c in listings.columns]

listings_clean = listings.drop(columns=cols_a_eliminar)

print(f"Columnas originales:  {listings.shape[1]}")
print(f"Columnas eliminadas:  {len(cols_a_eliminar)}")
print(f"Columnas restantes:   {listings_clean.shape[1]}")


Columnas originales:  106
Columnas eliminadas:  13
Columnas restantes:   93


In [4]:
# Columnas con más del 50% de nulos → eliminar
umbral = 50
nulos_pct = (listings_clean.isnull().sum() / len(listings_clean) * 100)
cols_muchos_nulos = nulos_pct[nulos_pct > umbral].index.tolist()

listings_clean = listings_clean.drop(columns=cols_muchos_nulos)

print(f"Columnas eliminadas por >50% nulos: {len(cols_muchos_nulos)}")
print(f"Columnas restantes: {listings_clean.shape[1]}")
print(f"\nColumnas eliminadas: {cols_muchos_nulos}")

Columnas eliminadas por >50% nulos: 8
Columnas restantes: 85

Columnas eliminadas: ['notes', 'access', 'house_rules', 'square_feet', 'weekly_price', 'monthly_price', 'license', 'jurisdiction_names']


In [5]:
# Numéricos → rellenar con mediana
cols_numericas = listings_clean.select_dtypes(include="number").columns
for col in cols_numericas:
    if listings_clean[col].isnull().sum() > 0:
        listings_clean[col] = listings_clean[col].fillna(listings_clean[col].median())

# Texto → rellenar con "Desconocido"
cols_texto = listings_clean.select_dtypes(include="object").columns
for col in cols_texto:
    if listings_clean[col].isnull().sum() > 0:
        listings_clean[col] = listings_clean[col].fillna("Desconocido")

nulos_restantes = listings_clean.isnull().sum().sum()
print(f"Valores nulos restantes: {nulos_restantes}")

Valores nulos restantes: 0


In [6]:
# Usamos IQR para detectar outliers
Q1 = listings_clean["price"].quantile(0.25)
Q3 = listings_clean["price"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = listings_clean[
    (listings_clean["price"] < limite_inferior) | 
    (listings_clean["price"] > limite_superior)
]

print(f"Q1: ${Q1:,.0f} | Q3: ${Q3:,.0f} | IQR: ${IQR:,.0f}")
print(f"Límite inferior: ${limite_inferior:,.0f}")
print(f"Límite superior: ${limite_superior:,.0f}")
print(f"Outliers detectados: {len(outliers)} ({len(outliers)/len(listings_clean)*100:.1f}%)")

# Eliminar outliers
listings_clean = listings_clean[
    (listings_clean["price"] >= limite_inferior) & 
    (listings_clean["price"] <= limite_superior)
]
print(f"\nFilas después de eliminar outliers: {len(listings_clean)}")

Q1: $1,394 | Q3: $3,319 | IQR: $1,925
Límite inferior: $-1,494
Límite superior: $6,206
Outliers detectados: 2082 (8.8%)

Filas después de eliminar outliers: 21647


In [7]:
listings_clean.to_csv("listings_clean.csv", index=False)

print(f"✅ Dataset exportado: listings_clean.csv")
print(f"   Filas:    {listings_clean.shape[0]}")
print(f"   Columnas: {listings_clean.shape[1]}")
print(f"\nRango de precios final:")
print(f"   Mínimo:  ${listings_clean['price'].min():,.0f}")
print(f"   Máximo:  ${listings_clean['price'].max():,.0f}")
print(f"   Promedio: ${listings_clean['price'].mean():,.0f}")
print(f"   Mediana: ${listings_clean['price'].median():,.0f}")

✅ Dataset exportado: listings_clean.csv
   Filas:    21647
   Columnas: 85

Rango de precios final:
   Mínimo:  $0
   Máximo:  $6,173
   Promedio: $2,261
   Mediana: $1,991


In [8]:
print("""
=== RESUMEN DEL PROCESO ETL - ETAPA 2 ===

EXTRACCIÓN
  - 3 datasets cargados: listings, calendar, reviews
  - listings original: 23.729 filas x 106 columnas

TRANSFORMACIÓN
  - Eliminadas 13 columnas irrelevantes/URLs
  - Eliminadas  8 columnas con >50% nulos
  - Imputados nulos numéricos con mediana
  - Imputados nulos texto con "Desconocido"
  - Eliminados 2.082 outliers de precio (método IQR)

RESULTADO FINAL
  - listings_clean.csv: 21.647 filas x 85 columnas
  - 0 valores nulos
  - Datos listos para visualización (Etapa 3)
""")


=== RESUMEN DEL PROCESO ETL - ETAPA 2 ===

EXTRACCIÓN
  - 3 datasets cargados: listings, calendar, reviews
  - listings original: 23.729 filas x 106 columnas

TRANSFORMACIÓN
  - Eliminadas 13 columnas irrelevantes/URLs
  - Eliminadas  8 columnas con >50% nulos
  - Imputados nulos numéricos con mediana
  - Imputados nulos texto con "Desconocido"
  - Eliminados 2.082 outliers de precio (método IQR)

RESULTADO FINAL
  - listings_clean.csv: 21.647 filas x 85 columnas
  - 0 valores nulos
  - Datos listos para visualización (Etapa 3)



In [9]:
print("""
=== RESUMEN DEL PROCESO ETL - ETAPA 2 ===

EXTRACCIÓN
  - 3 datasets cargados: listings, calendar, reviews
  - listings original: 23.729 filas x 106 columnas

TRANSFORMACIÓN
  - Eliminadas 13 columnas irrelevantes/URLs
  - Eliminadas  8 columnas con >50% nulos
  - Imputados nulos numéricos con mediana
  - Imputados nulos texto con "Desconocido"
  - Eliminados 2.082 outliers de precio (método IQR)

RESULTADO FINAL
  - listings_clean.csv: 21.647 filas x 85 columnas
  - 0 valores nulos
  - Datos listos para visualización (Etapa 3)
""")


=== RESUMEN DEL PROCESO ETL - ETAPA 2 ===

EXTRACCIÓN
  - 3 datasets cargados: listings, calendar, reviews
  - listings original: 23.729 filas x 106 columnas

TRANSFORMACIÓN
  - Eliminadas 13 columnas irrelevantes/URLs
  - Eliminadas  8 columnas con >50% nulos
  - Imputados nulos numéricos con mediana
  - Imputados nulos texto con "Desconocido"
  - Eliminados 2.082 outliers de precio (método IQR)

RESULTADO FINAL
  - listings_clean.csv: 21.647 filas x 85 columnas
  - 0 valores nulos
  - Datos listos para visualización (Etapa 3)



In [10]:
from pandasql import sqldf
pysql = lambda q: sqldf(q, globals())
print("✅ pandasql listo")

✅ pandasql listo


In [11]:
q1 = pysql("""
    SELECT neighbourhood_cleansed AS barrio,
           COUNT(*) AS cantidad_propiedades,
           ROUND(AVG(price), 0) AS precio_promedio
    FROM listings_clean
    GROUP BY neighbourhood_cleansed
    ORDER BY precio_promedio DESC
    LIMIT 10
""")
print("=== TOP 10 BARRIOS POR PRECIO PROMEDIO ===")
q1

=== TOP 10 BARRIOS POR PRECIO PROMEDIO ===


,barrio,cantidad_propiedades,precio_promedio
0,Villa Soldati,3,3762.0
1,Puerto Madero,144,3646.0
2,Palermo,6327,2630.0
3,Versalles,14,2589.0
4,Recoleta,3518,2484.0
5,Retiro,1091,2475.0
6,Nuñez,437,2226.0
7,San Telmo,680,2214.0
8,Belgrano,1022,2197.0
9,San Nicolas,1280,2184.0


In [12]:
q2 = pysql("""
    SELECT room_type,
           COUNT(*) AS cantidad,
           ROUND(AVG(price), 0) AS precio_promedio,
           ROUND(MIN(price), 0) AS precio_minimo,
           ROUND(MAX(price), 0) AS precio_maximo
    FROM listings_clean
    GROUP BY room_type
    ORDER BY cantidad DESC
""")
print("=== RESUMEN POR TIPO DE HABITACIÓN ===")
q2

=== RESUMEN POR TIPO DE HABITACIÓN ===


,room_type,cantidad,precio_promedio,precio_minimo,precio_maximo
0,Entire home/apt,16549,2549.0,0.0,6173.0
1,Private room,4349,1319.0,133.0,6173.0
2,Shared room,537,968.0,133.0,5377.0
3,Hotel room,212,2442.0,398.0,6107.0


In [13]:
q3 = pysql("""
    SELECT neighbourhood_cleansed AS barrio,
           ROUND(AVG(availability_365), 0) AS disponibilidad_promedio_anual,
           COUNT(*) AS propiedades
    FROM listings_clean
    GROUP BY neighbourhood_cleansed
    ORDER BY disponibilidad_promedio_anual DESC
    LIMIT 10
""")
print("=== BARRIOS CON MAYOR DISPONIBILIDAD ANUAL ===")
q3

=== BARRIOS CON MAYOR DISPONIBILIDAD ANUAL ===


,barrio,disponibilidad_promedio_anual,propiedades
0,Villa Soldati,303.0,3
1,Villa Real,268.0,7
2,Villa Riachuelo,261.0,2
3,Villa Lugano,241.0,3
4,Monserrat,224.0,934
5,Villa Santa Rita,223.0,24
6,Retiro,221.0,1091
7,Nueva Pompeya,219.0,12
8,Floresta,218.0,39
9,San Nicolas,215.0,1280
